In [ ]:
f_switch: float = 6.78e6     # 동작 주파수 [Hz]
delay_deg: float = 25        # ZVS 입력 위상 [deg]
R_load:    float = 50        # 뒷단 AC 등가부하 [ohm] (정류+벅부스트 입력)

# 코일 추출 + LCC-LCC 설계 @ 6.78 MHz

`last_678.csv`(정밀 Z) + `last_sweep.csv`(공진) + `last_field.csv`(재질별 손실).  
**구조**: 인버터 → 1차 LCC → 코일 → 2차 LCC → 정류 → 벅부스트 → 부하.  
뒷단 AC 부하 = `R_load`. 2차 Lf2가 이를 코일 최적부하로 변환 → 코일 최대효율 유지하며 출력 고전압·저전류.

단위: 인덕턴스 µH/nH, 커패시턴스 pF/nF.

## 0. 헬퍼 + 데이터 로드

In [ ]:
import csv, math, cmath
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

def fL(L):  return f'{L*1e6:.4f} µH' if abs(L)>=1e-6 else f'{L*1e9:.2f} nH'
def fC(Cv): return f'{Cv*1e9:.4f} nF' if abs(Cv)>=1e-9 else f'{Cv*1e12:.2f} pF'

f0=f_switch; w=2*math.pi*f0; phi_tgt=delay_deg
Vin=2*math.sqrt(2)/math.pi*320     # 풀브릿지 ±320V 기본파 RMS
P_tgt=500.0; RL=R_load

# 단자 Z (last_678.csv)
h,d=(lambda r:(r[0],r[1]))(list(csv.reader(open('last_678.csv'))))
def Zc(re,im): return float(d[h.index(re)])+1j*float(d[h.index(im)])
Z11=Zc('re(Zt(1_T1,1_T1)) []','im(Zt(1_T1,1_T1)) []')
Z22=Zc('re(Zt(2_T1,2_T1)) []','im(Zt(2_T1,2_T1)) []')
Z21=Zc('re(Zt(2_T1,1_T1)) []','im(Zt(2_T1,1_T1)) []'); Z12=Z21
R1,R2,wM=Z11.real,Z22.real,Z21.imag
print(f'Z11={Z11.real:.4f}{Z11.imag:+.3f}j  Z22={Z22.real:.4f}{Z22.imag:+.3f}j  Z21={Z21.real:.5f}{Z21.imag:+.4f}j')

## 1. 코일 파라미터 추출  (L + R1)∥C∥R2

- **C_self**: sweep 자기공진(SRF) + 저주파 L.
- **손실 분리**: `last_field.csv`의 구리손/비구리손 비율 r → R1(직렬·도체), R2(병렬·유전+코어).
- **L, R1, R2**: C 고정 + 손실비 r 제약 + 단자 Z 정확일치 (닫힌해).

In [ ]:
# 단자 유효값
L1e=Z11.imag/w; L2e=Z22.imag/w; Q1=Z11.imag/R1; Q2=Z22.imag/R2
M=wM/w; k=M/math.sqrt(L1e*L2e)

# SRF & C_self (sweep)
hs,ds=(lambda r:(r[0],r[1:]))(list(csv.reader(open('last_sweep.csv'))))
F=hs.index('Freq [MHz]')
def srf_Llow(imN):
    j=hs.index(imN); prev=None; Llow=None
    for r in ds:
        fr=float(r[F])*1e6; X=float(r[j])
        if Llow is None and fr>=0.5e6: Llow=X/(2*math.pi*fr)
        if prev and prev[1]>0 and X<0: f1,x1=prev; return f1+(fr-f1)*(x1/(x1-X)), Llow
        prev=(fr,X)
    return None,Llow
srf1,Ll1=srf_Llow('im(Zt(1_T1,1_T1)) []'); srf2,Ll2=srf_Llow('im(Zt(2_T1,2_T1)) []')
C1self=1/((2*math.pi*srf1)**2*Ll1); C2self=1/((2*math.pi*srf2)**2*Ll2)

# 손실비 (last_field.csv)
hl,dl=(lambda r:(r[0],r[1]))(list(csv.reader(open('last_field.csv'))))
def Lf_(n): return float(dl[hl.index(n)])
cu_tx=Lf_('loss_W_tx_ssw_coil_ssw_copper []')
nc_tx=(Lf_('loss_W_tx_ssw_coil_pcb_1_fr4 []')+Lf_('loss_W_tx_ssw_coil_pcb_2_fr4 []')
       +Lf_('loss_W_tx_mull_ferrite_sheet []')+Lf_('loss_W_tx_mull_ferrite_sheet_1 []')
       +Lf_('loss_W_tx_mull_ferrite_sheet_2 []')+Lf_('loss_W_tx_mull_ferrite_sheet_3 []'))
cu_rx=Lf_('loss_W_rx_ssw_coil_ssw_copper []')
nc_rx=Lf_('loss_W_rx_ssw_coil_pcb_1_fr4 []')+Lf_('loss_W_rx_ssw_coil_pcb_2_fr4 []')
r_tx=cu_tx/(cu_tx+nc_tx); r_rx=cu_rx/(cu_rx+nc_rx)

# 닫힌해 추출
def extract(Z,Cself,r):
    Y=1/Z; G=Y.real; B=Y.imag; a=w*Cself-B; rG=r*G
    L=a/(w*((rG)**2+a**2)); R1s=w*L*rG/a; R2p=1/(G*(1-r))
    return L,R1s,R2p
Lb1,R1s,R2p1=extract(Z11,C1self,r_tx)
Lb2,R2s,R2p2=extract(Z22,C2self,r_rx)

print('=== 코일 추출 @ 6.78 MHz ===')
print(f'Tx: L_eff={fL(L1e)}  Q={Q1:.0f}  | 모델 L={fL(Lb1)}  R1={R1s*1e3:.1f}mΩ(직렬/구리)  C={fC(C1self)}  R2={R2p1/1e3:.0f}kΩ(병렬/유전)  [구리손 {r_tx*100:.0f}%]')
print(f'Rx: L_eff={fL(L2e)}  Q={Q2:.0f}  | 모델 L={fL(Lb2)}  R1={R2s*1e3:.1f}mΩ(직렬/구리)  C={fC(C2self)}  R2={R2p2/1e3:.0f}kΩ(병렬/유전)  [구리손 {r_rx*100:.0f}%]')
print(f'결합: M={fL(M)}  k(단자)={k:.4f}  ω₀M={wM:.3f} Ω   SRF: Tx={srf1/1e6:.1f} Rx={srf2/1e6:.1f} MHz')

## 2. 링크 품질 & 최적부하

In [ ]:
FoM=wM**2/(R1*R2); eta_max=FoM/(1+math.sqrt(1+FoM))**2; Req=R2*math.sqrt(1+FoM)
print(f'kᵤ²={FoM:.0f}  →  링크 최대효율 η_max={eta_max*100:.1f}%')
print(f'코일 최적부하 R_eq,opt={Req:.3f} Ω')
print(f'뒷단 실제부하 R_L={RL:.0f} Ω  →  출력 {math.sqrt(P_tgt*RL):.0f} V / {math.sqrt(P_tgt/RL):.2f} A')

## 3. 2차 LCC 설계 (R_L → R_eq,opt 변환)

In [ ]:
Lf2=math.sqrt(Req*RL)/w; Cf2=1/(w**2*Lf2); C2=1/(w**2*(L2e-Lf2))
print(f'Lf2={fL(Lf2)}   Cf2={fC(Cf2)}   C2={fC(C2)}')

## 4. 전체 회로 해석기 (Z행렬 정확해)

In [ ]:
def C1_(Lf1): return 1/(w**2*(L1e-Lf1))
def solve(Lf1,Cf1):
    Y2=1j*w*Cf2+1/(1j*w*Lf2+RL); G=Y2/(1+Y2/(1j*w*C2))
    Zc_=Z11-Z12*Z21*G/(1+G*Z22); Zt=Zc_+1/(1j*w*C1_(Lf1))
    I1=Vin/(Zt*(1-w**2*Lf1*Cf1)+1j*w*Lf1)
    J2=G*Z21*I1/(1+G*Z22); I2=-J2
    VnP2=J2/Y2; IRL=VnP2/(1j*w*Lf2+RL); I_in=I1*(1+1j*w*Cf1*Zt)
    return dict(I1=I1,I2=I2,IRL=IRL,ICf2=VnP2*1j*w*Cf2,I_in=I_in,
                Pload=abs(IRL)**2*RL,Vload=abs(IRL)*RL,Zin=Vin/I_in,
                phase=math.degrees(cmath.phase(Vin/I_in)),
                eta=abs(IRL)**2*RL/(Vin*I_in.conjugate()).real,C1=C1_(Lf1),VnP1=I1*Zt)

## 5. 1차 해 — [∠Zin=delay_deg, P=500W]

In [ ]:
def resid(x):
    r=solve(x[0],x[1]); return [r['phase']-phi_tgt, r['Pload']-P_tgt]
Lf1,Cf1=fsolve(resid,[Vin/(w*17.0),1/(w**2*(Vin/(w*17.0)))])
r=solve(Lf1,Cf1); C1=r['C1']; Cf1_t=1/(w**2*Lf1)
print(f'수렴: ∠Zin={r["phase"]:.2f}°  Pload={r["Pload"]:.1f}W')

## 6. 설계 결과

In [ ]:
print('===== LCC-LCC 소자값 @ {:.2f} MHz (R_L={:.0f}Ω) ====='.format(f0/1e6,RL))
print(f'[1차]  Lf1={fL(Lf1)}   Cf1={fC(Cf1)} (동조 {fC(Cf1_t)}, {(Cf1/Cf1_t-1)*100:+.1f}% 디튠)   C1={fC(C1)}')
print(f'[2차]  Lf2={fL(Lf2)}   Cf2={fC(Cf2)}   C2={fC(C2)}')
print()
print('===== 동작점 =====')
print(f'∠Zin={r["phase"]:.2f}° (ZVS {phi_tgt}°)   P={r["Pload"]:.1f}W   η_link={r["eta"]*100:.1f}%')
print(f'출력: {r["Vload"]:.1f} V / {abs(r["IRL"]):.2f} A   (정류+벅부스트 입력)')
print(f'인버터: {abs(r["I_in"]):.2f} A (9A 한계, 여유 {9/abs(r["I_in"]):.1f}배)')
print(f'코일전류: I_L1={abs(r["I1"]):.1f} A  I_L2={abs(r["I2"]):.1f} A   (litz 정격 확인)')
print(f'2차 순환: Cf2={abs(r["ICf2"]):.1f} A')
print('\n--- 보상캡 전압/전류 (직병렬 설계용) ---')
for nm,Cv,I,V in [('C1',C1,abs(r['I1']),abs(r['I1'])/(w*C1)),
                  ('C2',C2,abs(r['I2']),abs(r['I2'])/(w*C2)),
                  ('Cf1',Cf1,abs(r['VnP1'])*w*Cf1,abs(r['VnP1'])),
                  ('Cf2',Cf2,abs(r['ICf2']),abs(r['ICf2'])/(w*Cf2))]:
    print(f'  {nm:4} {fC(Cv):>11}   I={I:5.1f} A   V={V:6.0f} V')

## 7. 주파수 응답 (ZVS 위상)

In [ ]:
fr=np.linspace(0.92,1.08,401)*f0; ph=[];po=[]
for fx in fr:
    ww=2*math.pi*fx
    Y2=1j*ww*Cf2+1/(1j*ww*Lf2+RL); G=Y2/(1+Y2/(1j*ww*C2))
    Zc_=Z11-Z12*Z21*G/(1+G*Z22); Zt=Zc_+1/(1j*ww*C1)
    I1=Vin/(Zt*(1-ww**2*Lf1*Cf1)+1j*ww*Lf1)
    J2=G*Z21*I1/(1+G*Z22); VnP2=J2/Y2; IRL=VnP2/(1j*ww*Lf2+RL)
    I_in=I1*(1+1j*ww*Cf1*Zt)
    ph.append(math.degrees(cmath.phase(Vin/I_in))); po.append(abs(IRL)**2*RL)
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(fr/1e6,ph); ax[0].axhline(phi_tgt,color='r',ls='--',label=f'{phi_tgt} deg'); ax[0].axvline(f0/1e6,color='k',ls=':'); ax[0].axhline(0,color='gray',lw=.5)
ax[0].set_title('Input phase angle(Zin)'); ax[0].set_xlabel('MHz'); ax[0].set_ylabel('deg'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(fr/1e6,po); ax[1].axhline(P_tgt,color='r',ls='--'); ax[1].axvline(f0/1e6,color='k',ls=':')
ax[1].set_title('Load power'); ax[1].set_xlabel('MHz'); ax[1].set_ylabel('W'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8. LTspice 넷리스트

코일 = 추출 풀모델 `(L_bare+R1)∥C_self∥R2` + 결합 K (L_bare 기준).

In [ ]:
kb=M/math.sqrt(Lb1*Lb2)   # bare-L 기준 결합계수
net=f'''* LCC-LCC @ {f0/1e6:.2f}MHz  Vin={Vin:.0f}Vrms  RL={RL:.0f}ohm  out~{r["Vload"]:.0f}V/{abs(r["IRL"]):.1f}A
Vin S 0 SINE(0 {Vin*math.sqrt(2):.1f} {f0:.0f})
* --- primary LCC ---
Lf1 S  P1 {Lf1:.4g}
Cf1 P1 0  {Cf1:.4g}
C1  P1 T1 {C1:.4g}
* Tx coil (L+R1)//Cself//R2
Ctx T1 0  {C1self:.4g}
R2t T1 0  {R2p1:.4g}
R1t T1 Tm {R1s:.4g}
Ltx Tm 0  {Lb1:.4g}
* Rx coil
Crx Rn 0  {C2self:.4g}
R2r Rn 0  {R2p2:.4g}
R1r Rn Rm {R2s:.4g}
Lrx Rm 0  {Lb2:.4g}
* --- secondary LCC ---
C2  Rn P2 {C2:.4g}
Cf2 P2 0  {Cf2:.4g}
Lf2 P2 Lo {Lf2:.4g}
RL  Lo 0  {RL:.4g}
K1  Ltx Lrx {kb:.4f}
.ac lin 401 {0.92*f0:.0f} {1.08*f0:.0f}
.end'''
print(net); open('LCC_LCC_6p78MHz.cir','w').write(net); print('\n-> LCC_LCC_6p78MHz.cir 저장')